# Redes Neuronales para Clasificación Binaria
### Deep Learning con Keras


Hoy vas a construir tu primera red neuronal. Usaremos un dataset médico real y Keras, la librería más usada para construir redes de forma intuitiva.

**Dataset:** Breast Cancer Wisconsin (UCI)  
**Librería:** Keras (sobre TensorFlow)

---
## Bloque 0 — Setup

Importamos todo lo que vamos a necesitar. Si alguna librería no está instalada, ejecuta:
```bash
pip install tensorflow scikit-learn matplotlib
```

Si estás usando Colab:
``!pip install tensorflow scikit-learn matplotlib``

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Silencia los logs verbose de TensorFlow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Fijamos semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print("✅ Setup completo")

---
## Bloque 1 — El dataset (~10 min)

### ¿Qué vamos a predecir?

El dataset **Breast Cancer Wisconsin** contiene mediciones de biopsias de tumores de mama. Para cada biopsia, el laboratorio extrae **30 medidas numéricas** (radio medio, textura, perímetro, área, suavidad...) y el diagnóstico real: `maligno` o `benigno`.

Nuestro objetivo: dado el perfil de una biopsia, predecir si el tumor es maligno o benigno.

| | |
|---|---|
| **Filas** | 569 |
| **Features** | 30 (todas numéricas) |
| **Target** | 0 = maligno, 1 = benigno |
| **Valores nulos** | Ninguno |

In [ ]:
# Cargamos el dataset directamente desde sklearn
# Es el mismo dataset que está en el UCI Machine Learning Repository
raw = load_breast_cancer()

# Construimos un DataFrame para explorarlo cómodamente
df = pd.DataFrame(raw.data, columns=raw.feature_names)
df['target'] = raw.target

print(f"Shape: {df.shape}")
print(f"\nClases: {dict(zip(raw.target_names, df['target'].value_counts().sort_index()))}")
df.head()

In [ ]:
# Estadísticas básicas — fíjate en los rangos muy distintos entre columnas
df.describe().round(2)

### 🔍 Observación importante

Fíjate en los rangos de las columnas: `mean radius` va de 6 a 28, mientras que `mean area` va de 143 a 2501.

Esto es un problema para las redes neuronales (y para cualquier modelo basado en gradientes): si una feature tiene valores 100 veces más grandes que otra, la red le dará artificialmente más peso solo por su escala, no por su importancia real.

**Solución: normalizar.** Con `StandardScaler` transformamos cada feature para que tenga media 0 y desviación estándar 1.

**¿Por qué no normalizamos con XGBoost?**  
> XGBoost usa árboles de decisión. Los árboles solo preguntan "¿es X > umbral?" — les da igual si X vale 5 o 5000, el orden relativo no cambia. Las redes neuronales en cambio calculan sumas ponderadas, donde la escala sí importa.

In [ ]:
# Separamos features y target
X = df.drop(columns=['target']).values
y = df['target'].values

# Split train / val / test — igual que en el notebook anterior
# Primero separamos test (20%), luego del resto separamos val (20%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
)

# Normalizamos — IMPORTANTE: el scaler se ajusta SOLO sobre train
# Si lo ajustáramos sobre todo el dataset, estaríamos usando información del futuro
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # ajusta Y transforma
X_val   = scaler.transform(X_val)        # solo transforma
X_test  = scaler.transform(X_test)       # solo transforma

print(f"Train:      {X_train.shape[0]} muestras")
print(f"Validación: {X_val.shape[0]} muestras")
print(f"Test:       {X_test.shape[0]} muestras")
print(f"\nMedia de X_train después de normalizar: {X_train.mean():.4f} (debería ser ~0)")
print(f"Std de X_train después de normalizar:   {X_train.std():.4f} (debería ser ~1)")

---
## Bloque 2 — Tu primera neurona 

### REPASO: ¿Qué es una neurona?

Una neurona artificial hace exactamente dos cosas:

1. **Combinación lineal:** suma todas sus entradas multiplicadas por sus pesos, más un bias

2. **Función de activación:** aplica una función no lineal al resultado

Recuerda: Los **pesos** `w` y el **bias** `b` son los parámetros que el modelo aprende durante el entrenamiento.


In [ ]:
# Una sola neurona, sin función de activación
model_sin_activacion = keras.Sequential([
    layers.Dense(1, input_shape=(30,))  # 1 neurona, 30 entradas
])

model_sin_activacion.summary()

El `summary()` te dice:
- **Output shape**: `(None, 1)` — None es el tamaño del batch, 1 es la salida
- **Param**: 31 — 30 pesos (uno por feature) + 1 bias

Esta neurona sin activación es exactamente una **regresión lineal**. Su salida puede ser cualquier número real: -∞ a +∞.

Para clasificación binaria, queremos que la salida sea una **probabilidad** entre 0 y 1.  
Necesitamos una función que comprima cualquier número real al rango (0, 1).

Esa función existe: se llama **Sigmoid** (o función logística).

In [ ]:
# Visualizamos la función Sigmoid
z = np.linspace(-6, 6, 200)
sigmoid = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, sigmoid, color='#e63946', linewidth=2.5, label='σ(z) = 1 / (1 + e⁻ᶻ)')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Umbral = 0.5')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=1, color='black', linewidth=0.5)
ax.fill_between(z, sigmoid, 0.5, where=(sigmoid > 0.5), alpha=0.1, color='green', label='Predice BENIGNO (1)')
ax.fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5), alpha=0.1, color='red', label='Predice MALIGNO (0)')
ax.set_xlabel('z (entrada)', fontsize=12)
ax.set_ylabel('σ(z)', fontsize=12)
ax.set_title('Función Sigmoid — convierte cualquier número real a probabilidad', fontsize=13)
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Ejemplos:")
for val in [-5, -2, 0, 2, 5]:
    s = 1 / (1 + np.exp(-val))
    print(f"  σ({val:>3}) = {s:.4f}")

### ¿Por qué Sigmoid en la última capa para clasificación binaria?

1. **Rango (0, 1)**: la salida se puede interpretar directamente como una probabilidad
2. **Interpretación**: `σ(z) = 0.8` significa "el modelo cree que hay un 80% de probabilidad de que sea benigno"
3. **Decisión final**: si `σ(z) ≥ 0.5` → clase 1 (benigno); si `σ(z) < 0.5` → clase 0 (maligno)



In [ ]:
# Una sola neurona CON función de activación Sigmoid
# Esto es equivalente a una regresión logística
model_una_neurona = keras.Sequential([
    layers.Dense(1, activation='sigmoid', input_shape=(30,))
])

model_una_neurona.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Arquitectura:")
model_una_neurona.summary()

In [ ]:
# Entrenamos esta neurona mínima — solo para ver qué puede aprender una sola neurona
hist_una = model_una_neurona.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    verbose=0
)

val_acc_una = hist_una.history['val_accuracy'][-1]
print(f"Accuracy en validación (1 neurona): {val_acc_una:.4f}")
print("→ Una sola neurona con sigmoid = regresión logística")

---
## Bloque 3 — La red completa

### Del perceptrón a la red multicapa (MLP)

Una sola neurona solo puede aprender **fronteras de decisión lineales**. Para capturar relaciones más complejas entre las features, necesitamos apilar múltiples capas.

Pero hay un problema: si apilas capas lineales, el resultado sigue siendo lineal con pesos combinados.

**La solución**: añadir una función de activación **no lineal** entre capas. La más popular es **ReLU**.

$$\text{ReLU}(z) = \max(0, z)$$

Parece simple, pero esta pequeña no-linealidad es lo que da a las redes su poder real.

In [ ]:
# Visualizamos ReLU vs Sigmoid
z = np.linspace(-4, 4, 200)
relu = np.maximum(0, z)
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, func, name, color, desc in zip(
    axes,
    [sigmoid, relu, tanh],
    ['Sigmoid', 'ReLU', 'Tanh'],
    ['#e63946', '#2a9d8f', '#e9c46a'],
    [
        'Salida: (0, 1)\n→ Última capa (clasificación binaria)',
        'Salida: [0, +∞)\n→ Capas ocultas (la más usada)',
        'Salida: (-1, 1)\n→ Alternativa a ReLU en ocultas'
    ]
):
    ax.plot(z, func, color=color, linewidth=2.5)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('z', fontsize=11)
    ax.text(0.05, 0.05, desc, transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax.grid(alpha=0.3)

plt.suptitle('Funciones de activación más comunes', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### ¿Por qué ReLU en capas ocultas?

- **Simple y eficiente**: solo necesita comparar con 0
- **No satura** para valores positivos: el gradiente no desaparece durante el entrenamiento
- **Funciona** en prácticamente todos los problemas como primera opción

### El patrón estándar de una MLP para clasificación binaria

```
Input (30 features)
       ↓
Dense(16) + ReLU    ← capa oculta 1: aprende combinaciones de features
       ↓
Dense(8)  + ReLU    ← capa oculta 2: aprende combinaciones de las anteriores
       ↓
Dense(1)  + Sigmoid ← capa de salida: convierte a probabilidad
```

In [ ]:
# Construimos la MLP completa
def construir_modelo(activacion_oculta='relu'):
    """Construye una MLP con la activación especificada en capas ocultas."""
    model = keras.Sequential([
        layers.Dense(16, activation=activacion_oculta, input_shape=(30,)),
        layers.Dense(8,  activation=activacion_oculta),
        layers.Dense(1,  activation='sigmoid')  # Siempre sigmoid aquí
    ], name=f'MLP_{activacion_oculta}')
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Modelo base con ReLU
modelo_base = construir_modelo('relu')
modelo_base.summary()

### Contando parámetros


- **Capa 1 (Dense 16)**: 30 entradas × 16 neuronas = 480 pesos + 16 biases = **496 params**
- **Capa 2 (Dense 8)**: 16 entradas × 8 neuronas = 128 pesos + 8 biases = **136 params**  
- **Capa 3 (Dense 1)**: 8 entradas × 1 neurona = 8 pesos + 1 bias = **9 params**


---
## Bloque 4 — Entrenamiento 

### ¿Qué es binary_crossentropy?

En el notebook de XGBoost usamos *log loss* durante el entrenamiento. Con redes para clasificación binaria usamos **binary crossentropy** — son exactamente lo mismo, solo el nombre cambia.

- Si la etiqueta real es 1 y el modelo predice 0.99 → pérdida muy baja (acertó con confianza)
- Si la etiqueta real es 1 y el modelo predice 0.01 → pérdida muy alta (erró con confianza)
- **Accuracy** es la métrica más legible, pero el modelo optimiza la pérdida, no el accuracy directamente.

### ¿Qué es Adam?

Adam es el optimizador más usado en deep learning. Se encarga de ajustar los pesos en cada paso del entrenamiento. No necesitas entender su interior ahora — lo importante es que funciona bien como punto de partida en casi todos los problemas.

In [ ]:
# Entrenamos el modelo base
historia = modelo_base.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    verbose=0  # Silenciamos el output epoch a epoch
)

print("Entrenamiento completado")
print(f"Accuracy final en train:      {historia.history['accuracy'][-1]:.4f}")
print(f"Accuracy final en validación: {historia.history['val_accuracy'][-1]:.4f}")

In [ ]:
def plot_historia(historia, titulo='Entrenamiento'):
    """Visualiza las curvas de loss y accuracy durante el entrenamiento."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Loss
    axes[0].plot(historia.history['loss'], label='Train', color='#2a9d8f', linewidth=2)
    axes[0].plot(historia.history['val_loss'], label='Validación', color='#e63946', linewidth=2)
    axes[0].set_title('Pérdida (Binary Crossentropy)', fontsize=13)
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(historia.history['accuracy'], label='Train', color='#2a9d8f', linewidth=2)
    axes[1].plot(historia.history['val_accuracy'], label='Validación', color='#e63946', linewidth=2)
    axes[1].set_title('Accuracy', fontsize=13)
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(titulo, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_historia(historia, titulo='Modelo base — ReLU en capas ocultas, Sigmoid en salida')

### ¿Qué ves en estas curvas?

- La **pérdida baja** con el tiempo en ambos conjuntos: el modelo está aprendiendo
- El **accuracy sube** de forma simétrica
- Train y val van juntos: no hay overfitting severo en este caso

¿Qué pasaba si la curva de validación empezara a subir mientras la de train sigue bajando?

---
## Bloque 5 — Evaluación

In [ ]:
# Funciones de métricas — las mismas que en el notebook de XGBoost
def calc_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def calc_precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def calc_recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def calc_f1_score(y_true, y_pred):
    p = calc_precision(y_true, y_pred)
    r = calc_recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def compute_metrics(y_true, y_pred, nombre='Modelo'):
    print(f"\n📊 Métricas — {nombre}")
    print("-" * 40)
    print(f"  Accuracy:  {calc_accuracy(y_true, y_pred):.4f}")
    print(f"  Precision: {calc_precision(y_true, y_pred):.4f}")
    print(f"  Recall:    {calc_recall(y_true, y_pred):.4f}")
    print(f"  F1 Score:  {calc_f1_score(y_true, y_pred):.4f}")

print("✅ Funciones de métricas cargadas")

In [ ]:
# Predicciones sobre validación
# modelo.predict() devuelve probabilidades (salida del Sigmoid)
y_prob_val = modelo_base.predict(X_val, verbose=0).flatten()

print("Primeras 5 probabilidades predichas:")
print(y_prob_val[:5].round(4))
print("\nEtiquetas reales:")
print(y_val[:5])

# Convertimos probabilidades a clases con umbral 0.5
y_pred_val = (y_prob_val >= 0.5).astype(int)

compute_metrics(y_val, y_pred_val, nombre='MLP ReLU — Validación')

In [ ]:
# Evaluación final en TEST — RECUERDA !! solo la tocamos UNA VEZ al final
y_prob_test = modelo_base.predict(X_test, verbose=0).flatten()
y_pred_test = (y_prob_test >= 0.5).astype(int)

compute_metrics(y_test, y_pred_test, nombre='MLP ReLU — Test (evaluación final)')

In [ ]:
# Matriz de confusión visual
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_test)
etiquetas = raw.target_names  # ['malignant', 'benign']

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(etiquetas, fontsize=11)
ax.set_yticklabels(etiquetas, fontsize=11)
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
ax.set_title('Matriz de confusión — Test', fontsize=13)

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=18, fontweight='bold',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

print("\n En diagnóstico médico, el Recall importa más que la Precision:")
print("   Un falso negativo (no detectar un cáncer real) es mucho más grave")
print("   que un falso positivo (alarmar a alguien sano).")

---
## Bloque 6 — Experimentación

### Experimento 1: ¿Qué pasa si no usamos ninguna activación en las capas ocultas?


**¿Qué esperas que pase con el accuracy?**

In [ ]:
# Experimento 1: Sin activación en capas ocultas
modelo_sin_act = keras.Sequential([
    layers.Dense(16, activation=None, input_shape=(30,)),  # Sin activación
    layers.Dense(8,  activation=None),                     # Sin activación
    layers.Dense(1,  activation='sigmoid')                 # Sigmoid solo al final
], name='sin_activacion')

modelo_sin_act.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

hist_sin = modelo_sin_act.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100, batch_size=32, verbose=0
)

y_pred_sin = (modelo_sin_act.predict(X_val, verbose=0).flatten() >= 0.5).astype(int)

print("=" * 50)
print("COMPARACIÓN: Con ReLU vs Sin activación")
print("=" * 50)
compute_metrics(y_val, y_pred_val, nombre='Con ReLU (modelo base)')
compute_metrics(y_val, y_pred_sin, nombre='Sin activación en ocultas')

<details>
<summary><b>Respuesta — ¿qué está pasando?</b> (haz clic para ver)</summary>

La red sin activaciones en capas ocultas tiene un rendimiento similar a una **regresión logística** (una sola neurona con sigmoid), porque matemáticamente es equivalente.

Cuando apilas capas lineales:
$$\text{capa2}(\text{capa1}(x)) = W_2(W_1 x + b_1) + b_2 = \underbrace{(W_2 W_1)}_{\text{nueva W}} x + \underbrace{(W_2 b_1 + b_2)}_{\text{nuevo b}}$$

El resultado es siempre una transformación lineal. Tres capas lineales = una capa lineal. Los 641 parámetros no te compran más expresividad que los 31 de la regresión logística.

**La función de activación es lo que convierte una red profunda en algo más potente que una red superficial.**

</details>

---

### Experimento 2: ¿Qué pasa si quitamos el Sigmoid de la capa de salida?

**¿Qué esperas que pase?**

In [ ]:
# Experimento 3: Sin sigmoid en la capa de salida
modelo_sin_sigmoid = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(30,)),
    layers.Dense(8,  activation='relu'),
    layers.Dense(1,  activation=None)  # Sin activación en salida
], name='sin_sigmoid_salida')

modelo_sin_sigmoid.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

hist_sin_sig = modelo_sin_sigmoid.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100, batch_size=32, verbose=0
)

# ¿Qué devuelve este modelo?
salidas_raw = modelo_sin_sigmoid.predict(X_val[:5], verbose=0).flatten()
print("Salidas del modelo SIN sigmoid (primeras 5):")
print(salidas_raw.round(3))
print("\n¿Son probabilidades? Pueden ser cualquier número real...")
print(f"Min: {salidas_raw.min():.3f}, Max: {salidas_raw.max():.3f}")

# Para clasificar, podemos usar umbral 0 en lugar de 0.5
y_pred_sin_sig = (salidas_raw >= 0).astype(int)

# Loss curves
plot_historia(hist_sin_sig, titulo='Sin sigmoid en salida — ¿qué pasa con la loss?')

<details>
<summary><b>Respuesta — ¿qué está pasando?</b> (haz clic para ver)</summary>

Sin sigmoid en la salida, el modelo devuelve **logits** — números reales sin restricción de rango. La función `binary_crossentropy` de Keras está diseñada para recibir probabilidades (0 a 1).

Keras en realidad tiene dos modos:
- `binary_crossentropy` + `sigmoid` en la salida: el modo estándar
- `binary_crossentropy(from_logits=True)` + sin activación: matemáticamente equivalente pero numéricamente más estable

Si usas `binary_crossentropy` sin `from_logits=True` y sin sigmoid, **el modelo puede entrar en inestabilidad numérica** — verás la loss con valores extraños o NaN.

**Regla práctica**: para clasificación binaria, elige uno de estos dos patrones y no los mezcles:
1. `Dense(1, activation='sigmoid')` + `loss='binary_crossentropy'`
2. `Dense(1)` (sin activación) + `loss=BinaryCrossentropy(from_logits=True)`

</details>

---
## Bloque 7 — Resumen


### Lo que has aprendido hoy

| Concepto | Idea clave |
|---|---|
| **Neurona** | Combinación lineal + función de activación |
| **Sigmoid** | Comprime cualquier número a (0, 1) → probabilidad |
| **ReLU** | max(0, z) — introduce no linealidad en capas ocultas |
| **Sin activación** | Equivale a una sola capa lineal, sin importar la profundidad |
| **MLP** | Capas densas apiladas: Dense + ReLU → ... → Dense + Sigmoid |
| **binary_crossentropy** | La función de pérdida para clasificación binaria |
| **Adam** | El optimizador estándar para empezar |
